In [1]:
import pandas as pd
import numpy as np
from io import StringIO

with open("chronic_kidney_disease.arff", "r") as f:
    lines = f.readlines()

data_start = [i for i, line in enumerate(lines) if line.strip().upper() == '@DATA'][0] + 1
data_lines = lines[data_start:]
columns = [line.split()[1] for line in lines if line.strip().upper().startswith('@ATTRIBUTE')]

data_str = ''.join(data_lines)
data_df = pd.read_csv(StringIO(data_str), header=None, names=columns, na_values='?', on_bad_lines='skip')
data_df.columns = data_df.columns.str.replace("'", "").str.strip()

# 공백/탭 제거
obj_cols = data_df.select_dtypes(include='object').columns
for col in obj_cols:
    data_df[col] = data_df[col].str.strip()

print(data_df['dm'].value_counts())
print(data_df['class'].value_counts())

dm
no     260
yes    135
Name: count, dtype: int64
class
ckd       248
notckd    149
Name: count, dtype: int64


In [2]:
# pcv, wbcc, rbcc 숫자로 변환
for col in ['pcv', 'wbcc', 'rbcc']:
    data_df[col] = pd.to_numeric(data_df[col], errors='coerce')

# 수치형 컬럼 중앙값으로 채우기
num_cols = data_df.select_dtypes(include='float64').columns
data_df[num_cols] = data_df[num_cols].fillna(data_df[num_cols].median())

# 범주형 컬럼 최빈값으로 채우기
obj_cols = data_df.select_dtypes(include='object').columns
for col in obj_cols:
    data_df[col] = data_df[col].fillna(data_df[col].mode()[0])

# 범주형 컬럼 인코딩
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in obj_cols:
    data_df[col] = le.fit_transform(data_df[col])

print(data_df['class'].value_counts())
print(data_df.isnull().sum().sum())

class
0    248
1    149
Name: count, dtype: int64
0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X = data_df.drop('class', axis=1)
y = data_df['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("정확도:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

정확도: 0.9625
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        51
           1       1.00      0.90      0.95        29

    accuracy                           0.96        80
   macro avg       0.97      0.95      0.96        80
weighted avg       0.96      0.96      0.96        80



In [4]:
def predict_ckd(age, bp, sg, al, su, rbc, pc, pcc, ba, bgr, bu, sc, sod, pot, hemo, pcv, wbcc, rbcc, htn, dm, cad, appet, pe, ane):
    
    patient = pd.DataFrame([[age, bp, sg, al, su, rbc, pc, pcc, ba, bgr, bu, sc, sod, pot, hemo, pcv, wbcc, rbcc, htn, dm, cad, appet, pe, ane]], 
                           columns=X.columns)
    
    prediction = model.predict(patient)[0]
    probability = model.predict_proba(patient)[0]
    
    if prediction == 0:
        result = "CKD (만성신장질환)"
    else:
        result = "정상 (Non-CKD)"
    
    print(f"예측 결과: {result}")
    print(f"CKD 확률: {probability[0]*100:.1f}%")
    print(f"정상 확률: {probability[1]*100:.1f}%")

# 예시 환자 (정상 수치)
predict_ckd(
    age=50, bp=80, sg=1.020, al=0, su=0,
    rbc=1, pc=1, pcc=0, ba=0,
    bgr=100, bu=20, sc=1.0,
    sod=140, pot=4.5, hemo=15.0,
    pcv=44, wbcc=7800, rbcc=5.2,
    htn=0, dm=0, cad=0, appet=1, pe=0, ane=0
)

예측 결과: 정상 (Non-CKD)
CKD 확률: 14.0%
정상 확률: 86.0%


In [5]:
# 예시 환자 (CKD 수치)
predict_ckd(
    age=60, bp=100, sg=1.005, al=4, su=3,
    rbc=0, pc=0, pcc=1, ba=1,
    bgr=200, bu=80, sc=5.0,
    sod=120, pot=6.0, hemo=8.0,
    pcv=25, wbcc=12000, rbcc=3.0,
    htn=1, dm=1, cad=1, appet=0, pe=1, ane=1
)

예측 결과: CKD (만성신장질환)
CKD 확률: 100.0%
정상 확률: 0.0%


In [6]:
!pip install streamlit

In [7]:
streamlit run app.py

SyntaxError: invalid syntax (3737097518.py, line 1)